In [ ]:
# @title IndoWordNet Kannada Interface
import json
import pandas as pd
from IPython.display import display, HTML

# 1. Load and process your dataset
try:
    # Adjust path if necessary
    df = pd.read_csv('/content/drive/MyDrive/IndoWordNet_Project/word.csv')

    word_data = {}
    for _, row in df.iterrows():
        word_key = str(row['word']).strip()
        entry = {
            'synset_id': str(row['synset_id']).replace('.0', ''),
            'pos': str(row['pos']).upper(),
            'gloss': str(row['gloss']),
            'examples': str(row['examples']),
            'synonyms': str(row['synonyms'])
        }
        if word_key not in word_data:
            word_data[word_key] = []
        word_data[word_key].append(entry)

    word_json_str = json.dumps(word_data, ensure_ascii=False)
except Exception as e:
    word_json_str = "{}"
    print(f"Error processing data: {e}")

html = f"""
<!DOCTYPE html>
<html>
<head>
<style>
    :root {{
        --bg-color: #ffffff;
        --text-color: #333333;
        --header-bg: #7fffd4;
        --box-bg: #ffffff;
        --border-color: #ccc;
        --shadow-color: #888888;
        --input-border: #9ec5ff;
        --row-hover: #f1f1f1;
    }}

    [data-theme="dark"] {{
        --bg-color: #121212;
        --text-color: #e0e0e0;
        --header-bg: #004d40;
        --box-bg: #1e1e1e;
        --border-color: #444444;
        --shadow-color: #000000;
        --input-border: #3d5afe;
        --row-hover: #2c2c2c;
    }}

    body {{ margin: 0; font-family: Arial, sans-serif; background-color: var(--bg-color); color: var(--text-color); transition: 0.3s; }}
.header {{
        background: var(--header-bg);
        padding: 10px 40px;
        border-bottom: 1px solid var(--border-color);
        display: flex;
        justify-content: space-between;
        align-items: center;
        color: var(--text-color);
    }}

      .header h1 {{ margin: 0; font-size: 32px; font-family: serif; color: var(--text-color); }}

    /* Dark Mode Toggle */
    .switch-container {{ display: flex; align-items: center; gap: 10px; font-weight: bold; font-size: 14px; }}
    .switch {{ position: relative; display: inline-block; width: 40px; height: 20px; }}
    .switch input {{ opacity: 0; width: 0; height: 0; }}
    .slider {{ position: absolute; cursor: pointer; top: 0; left: 0; right: 0; bottom: 0; background-color: #ccc; transition: .4s; border-radius: 20px; }}
    .slider:before {{ position: absolute; content: ""; height: 14px; width: 14px; left: 3px; bottom: 3px; background-color: white; transition: .4s; border-radius: 50%; }}
    input:checked + .slider {{ background-color: #2196F3; }}
    input:checked + .slider:before {{ transform: translateX(20px); }}

    .main-wrapper {{ width: 1300px; margin: 50px auto; display: flex; flex-direction: column; align-items: center; }}
    .search-box {{ background: var(--box-bg); border: 1px solid var(--border-color); box-shadow: 10px 10px 5px var(--shadow-color); padding: 15px 25px; width: 1200px; position: relative; min-height: 80px; }}
    .input-row {{ display: flex; align-items: center; gap: 10px; margin-bottom: 15px; }}
    .lang-select {{ width: 160px; padding: 6px; border: 1px solid var(--border-color); background: var(--box-bg); color: var(--text-color); }}
    .input-area-wrapper {{ flex: 1; }}
    #searchWord {{ width: 100%; padding: 8px; font-size: 16px; border: 1px solid var(--input-border); outline: none; box-sizing: border-box; background: var(--box-bg); color: var(--text-color); }}
    .btn-submit {{ background: #007bff; color: white; border: none; padding: 9px 25px; font-weight: bold; cursor: pointer; border-radius: 3px; }}

    #suggestions {{ width: 200px; background: var(--box-bg); border: 1px solid var(--border-color); display: none; z-index: 3000; box-shadow: 0 4px 8px rgba(0,0,0,0.2); max-height: 300px; overflow-y: auto; position: absolute; left: 170px; top: 0; }}
    .sug-row {{ padding: 8px; cursor: pointer; font-size: 14px; font-weight: bold; border-bottom: 1px solid var(--border-color); color: var(--text-color); }}
    .sug-row:hover {{ background: var(--row-hover); }}

    .vk-container {{ display: flex; flex-direction: column; align-items: center; position: relative; }}
    .vk-toggle-btn {{ width: 450px; padding: 6px 0; background: var(--box-bg); border: 1px solid var(--border-color); cursor: pointer; font-size: 13px; text-align: center; color: var(--text-color); }}
    .keyboard-panel {{ width: 450px; background: var(--box-bg); border: 1px solid var(--text-color); padding: 5px; display: none; box-sizing: border-box; box-shadow: 5px 5px 5px var(--shadow-color); position: absolute; top: 100%; z-index: 2000; }}

    #results-area {{ width: 1250px; margin-top: 20px; display: none; }}
    .result-block {{ border: 1px solid var(--border-color); box-shadow: 5px 5px 5px var(--shadow-color); background: var(--box-bg); padding: 10px; margin-bottom: 20px; }}
    .res-header {{ font-weight: bold; border-bottom: 1px solid var(--border-color); padding: 5px; color: var(--text-color); }}
    .res-table {{ width: 100%; border-collapse: collapse; margin-top: 5px; color: var(--text-color); }}
    .res-table tr {{ border-bottom: 1px solid var(--border-color); }}
    .res-table td {{ padding: 8px; vertical-align: top; font-size: 14px; }}
    .label-col {{ font-weight: bold; width: 180px; text-align: left; padding-left: 20px !important; }}
    .syn-link-item {{ color: #00a2ff; text-decoration: none; cursor: pointer; font-weight: bold; }}

    .kb-top button {{ padding: 2px; background: var(--box-bg); border: 1px solid var(--border-color); color: var(--text-color); cursor: pointer; font-weight: bold; flex: 1; }}
        table.kb-table {{ border-collapse: collapse; width: 100%; border: 1px solid #000; }}

    table.kb-table td {{ width: 28px; height: 28px; border: 1px solid var(--border-color); text-align: center; cursor: pointer; color: #00a2ff; font-size: 18px; background: var(--box-bg); }}

    .nav-buttons {{ display: flex; justify-content: center; gap: 20px; margin-top: 15px; }}
    .nav-btn {{ background: #007bff; color: white; border: none; padding: 8px 20px; font-weight: bold; cursor: pointer; border-radius: 4px; }}
</style>
</head>
<body>

<div class="header">
    <h1>Kannada WordNet</h1>
    <div class="switch-container">
        <span>Dark Mode</span>
        <label class="switch">
            <input type="checkbox" id="themeToggle" onclick="window.toggleTheme()">
            <span class="slider"></span>
        </label>
    </div>
</div>

<div class="main-wrapper">
    <div class="search-box">
        <div class="input-row">
            <select class="lang-select"><option>ಕನ್ನಡ (Kannada)</option></select>
            <div class="input-area-wrapper">
                <input id="searchWord" type="text" oninput="window.handleSug(this.value)" placeholder="Search Word" autocomplete="off" onkeypress="if(event.key==='Enter'){{window.searchData();}}">
            </div>
            <button class="btn-submit" onclick="window.searchData()">Submit</button>
        </div>
        <div class="bottom-row" style="display: flex; position: relative; justify-content: center;">
            <div id="suggestions"></div>
            <div class="vk-container">
                <div class="vk-toggle-btn" onclick="window.toggleKB()">Virtual Keyboard</div>
                <div id="keyboard" class="keyboard-panel">
                    <div class="kb-top" style="display:flex;">
                        <button onclick="window.addCh(' ')">Space</button>
                        <button onclick="window.backCh()">Backspace</button>
                        <button onclick="window.resetAll()">Reset</button>
                    </div>
                    <table class="kb-table">
                        <tr><td colspan="1"></td><td onclick="window.addCh('ಅ')">ಅ</td><td onclick="window.addCh('ಆ')">ಆ</td><td onclick="window.addCh('ಇ')">ಇ</td><td onclick="window.addCh('ಈ')">ಈ</td><td onclick="window.addCh('ಉ')">ಉ</td><td onclick="window.addCh('ಊ')">ಊ</td><td onclick="window.addCh('ಋ')">ಋ</td><td onclick="window.addCh('ಎ')">ಎ</td><td onclick="window.addCh('ಏ')">ಏ</td><td onclick="window.addCh('ಐ')">ಐ</td><td onclick="window.addCh('ಒ')">ಒ</td><td onclick="window.addCh('ಓ')">ಓ</td><td onclick="window.addCh('ಔ')">ಔ</td><td colspan="1"></td></tr>
                        <tr><td onclick="window.addCh('ಾ')">ಾ</td><td onclick="window.addCh('ಿ')">ಿ</td><td onclick="window.addCh('ೀ')">ೀ</td><td onclick="window.addCh('ು')">ು</td><td onclick="window.addCh('ೂ')">ೂ</td><td onclick="window.addCh('ೃ')">ೃ</td><td onclick="window.addCh('ೆ')">ೆ</td><td onclick="window.addCh('ೇ')">ೇ</td><td onclick="window.addCh('ೈ')">ೈ</td><td onclick="window.addCh('ೊ')">ೊ</td><td onclick="window.addCh('ೋ')">ೋ</td><td onclick="window.addCh('ೌ')">ೌ</td><td onclick="window.addCh('್')">್</td><td onclick="window.addCh('ಂ')">ಂ</td><td onclick="window.addCh('ಃ')">ಃ</td></tr>
                        <tr><td colspan="5"></td><td onclick="window.addCh('ಕ')">ಕ</td><td onclick="window.addCh('ಖ')">ಖ</td><td onclick="window.addCh('ಗ')">ಗ</td><td onclick="window.addCh('ಘ')">ಘ</td><td onclick="window.addCh('ಙ')">ಙ</td><td colspan="10"></td></tr>
                        <tr><td colspan="5"></td><td onclick="window.addCh('ಚ')">ಚ</td><td onclick="window.addCh('ಛ')">ಛ</td><td onclick="window.addCh('ಜ')">ಜ</td><td onclick="window.addCh('ಝ')">ಝ</td><td onclick="window.addCh('ಞ')">ಞ</td><td colspan="10"></td></tr>
                        <tr><td colspan="5"></td><td onclick="window.addCh('ಟ')">ಟ</td><td onclick="window.addCh('ಠ')">ಠ</td><td onclick="window.addCh('ಡ')">ಡ</td><td onclick="window.addCh('ಢ')">ಢ</td><td onclick="window.addCh('ಣ')">ಣ</td><td colspan="10"></td></tr>
                        <tr><td colspan="5"></td><td onclick="window.addCh('ತ')">ತ</td><td onclick="window.addCh('ಥ')">ಥ</td><td onclick="window.addCh('ದ')">ದ</td><td onclick="window.addCh('ಧ')">ಧ</td><td onclick="window.addCh('ನ')">ನ</td><td colspan="10"></td></tr>
                        <tr><td colspan="5"></td><td onclick="window.addCh('ಪ')">ಪ</td><td onclick="window.addCh('ಫ')">ಫ</td><td onclick="window.addCh('ಬ')">ಬ</td><td onclick="window.addCh('ಭ')">ಭ</td><td onclick="window.addCh('ಮ')">ಮ</td><td colspan="10"></td></tr>
                        <tr><td colspan="3"></td><td onclick="window.addCh('ಯ')">ಯ</td><td onclick="window.addCh('ರ')">ರ</td><td onclick="window.addCh('ಲ')">ಲ</td><td onclick="window.addCh('ವ')">ವ</td><td onclick="window.addCh('ಶ')">ಶ</td><td onclick="window.addCh('ಷ')">ಷ</td><td onclick="window.addCh('ಸ')">ಸ</td><td onclick="window.addCh('ಹ')">ಹ</td><td onclick="window.addCh('ಳ')">ಳ</td><td colspan="6"></td></tr>
                    </table>
                </div>
            </div>
        </div>
    </div>
    <div id="results-area"></div>
</div>

<script>
window.dataMap = {word_json_str};
window.searchHistory = [];
window.historyIndex = -1;

window.toggleTheme = function() {{
    const isDark = document.getElementById('themeToggle').checked;
    document.documentElement.setAttribute('data-theme', isDark ? 'dark' : 'light');
}};

window.searchSynonym = function(word) {{
    document.getElementById("searchWord").value = word;
    window.searchData(true);
}};

window.handleSug = function(val) {{
    let s = document.getElementById("suggestions");
    s.innerHTML = "";
    if(!val) {{ s.style.display="none"; return; }}
    let matches = Object.keys(window.dataMap).filter(k => k.startsWith(val)).slice(0,10);
    if(matches.length === 0) {{ s.style.display="none"; return; }}
    matches.forEach(m => {{
        let d = document.createElement("div"); d.className="sug-row"; d.textContent=m;
        d.onclick = () => {{ document.getElementById("searchWord").value=m; s.style.display="none"; window.searchData(); }};
        s.appendChild(d);
    }});
    s.style.display="block";
}};

window.toggleKB = function() {{
    let k = document.getElementById("keyboard");
    k.style.display = (k.style.display === "none" || k.style.display === "") ? "block" : "none";
}};

window.addCh = function(c) {{ let i=document.getElementById("searchWord"); i.value+=c; window.handleSug(i.value); }};
window.backCh = function() {{ let i=document.getElementById("searchWord"); i.value=i.value.slice(0,-1); window.handleSug(i.value); }};
window.resetAll = function() {{
    document.getElementById("searchWord").value="";
    document.getElementById("suggestions").style.display="none";
    document.getElementById("results-area").innerHTML="";
}};

window.searchData = function(addToHistory = true) {{
    document.getElementById("keyboard").style.display = "none";
    document.getElementById("suggestions").style.display = "none";

    let query = document.getElementById("searchWord").value.trim();
    if (addToHistory && query !== "") {{
        window.searchHistory = window.searchHistory.slice(0, window.historyIndex + 1);
        window.searchHistory.push(query);
        window.historyIndex++;
    }}
    let resultsArea = document.getElementById("results-area");
    resultsArea.innerHTML = "";

    if (window.dataMap[query]) {{
        resultsArea.style.display = "block";
        let synsets = window.dataMap[query];
        synsets.forEach((entry, index) => {{
            let synList = entry.synonyms.split(',').map(s => s.trim());
            let hyperlinkedSynonyms = synList.map(s => `<span class="syn-link-item" onclick="window.searchSynonym('${{s}}')">${{s}}</span>`).join(', ');
            let block = document.createElement("div");
            block.className = "result-block";
            block.innerHTML = `
                <div class="res-header"><span>Number of Synset for "${{query}}" : ${{synsets.length}}</span></div>
                <table class="res-table">
                    <tr><td class="label-col">Synset ID</td><td class="colon-col">:</td><td>${{entry.synset_id}}</td><td class="label-col">POS</td><td class="colon-col">:</td><td>${{entry.pos}}</td></tr>
                    <tr><td class="label-col">Gloss</td><td class="colon-col">:</td><td colspan="4">${{entry.gloss}}</td></tr>
                    <tr><td class="label-col">Example statement</td><td class="colon-col">:</td><td colspan="4">"${{entry.examples.replaceAll(query, '<span style="background:yellow;color:black;font-weight:bold;">'+query+'</span>')}}"</td></tr>
                    <tr><td class="label-col">Synonyms</td><td class="colon-col">:</td><td colspan="4">${{hyperlinkedSynonyms}}</td></tr>
                </table>`;
            resultsArea.appendChild(block);
        }});

        let navDiv = document.createElement("div");
        navDiv.className = "nav-buttons";
        navDiv.innerHTML = '<button class="nav-btn" onclick="window.goBack()">Previous</button><button class="nav-btn" onclick="window.goForward()">Next</button>';
        resultsArea.appendChild(navDiv);
        window.updateNavButtons();
    }} else {{

        resultsArea.style.display = "block";
        let similarWords = Object.keys(window.dataMap)
            .filter(k => {{
                return k.includes(query) ||
                       query.includes(k) ||
                       k.substring(0, query.length-1) === query.substring(0, query.length-1);
            }})
            .slice(0, 8); // Limit to top 8 suggestions

        let suggestionHtml = similarWords.length > 0
            ? `<div style="margin-top:10px; color: var(--text-color);">Did you mean: ${{similarWords.map(w => `<span class="syn-link-item" onclick="window.searchSynonym('${{w}}')">${{w}}</span>`).join(', ')}}?</div>`
            : "";

        resultsArea.innerHTML = `
            <div style="background:#f8d7da; color:#721c24; padding:20px; text-align:center; border: 1px solid #f5c6cb; border-radius:5px; width:600px; margin: 0 auto; position:relative;">
                <div>Word <b>${{query}}</b> not found in WordNet</div>
                ${{suggestionHtml}}
                <span style="position:absolute; top:5px; right:10px; cursor:pointer; font-weight:bold;" onclick="document.getElementById('results-area').style.display='none'">X</span>
            </div>
        `;
     }}
}};

window.goBack = function() {{ if (window.historyIndex > 0) {{ window.historyIndex--; document.getElementById("searchWord").value = window.searchHistory[window.historyIndex];window.searchData(false);  }} }};
window.goForward = function() {{ if (window.historyIndex < window.searchHistory.length - 1) {{ window.historyIndex++; document.getElementById("searchWord").value = window.searchHistory[window.historyIndex];window.searchData(false);  }} }};

window.updateNavButtons = function() {{
    let prevBtn = document.querySelector(".nav-btn:first-child");
    let nextBtn = document.querySelector(".nav-btn:last-child");
    if (prevBtn) prevBtn.style.display = (window.historyIndex > 0) ? "inline-block" : "none";
    if (nextBtn) nextBtn.style.display = (window.historyIndex < window.searchHistory.length - 1) ? "inline-block" : "none";
}};
</script>
</body>
</html>
"""

display(HTML(html))